<a href="https://colab.research.google.com/github/Bezawit-cloud/efficient-llm-finetuning/blob/main/notebooks/cloud_run.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Adaptive Data Selection & Curriculum Learning for Compute-Efficient LLM Fine-Tuning
## Cloud GPU Execution & Experiment Orchestration Notebook

This notebook orchestrates the complete 2×2 experiment matrix and ablations on a Cloud GPU instance (Google Colab, RunPod, Lambda Labs, or Kaggle):

| Exp | Selection | Ordering | Description |
|:---|:---|:---|:---|
| **E1** | 100% (Full) | Random | Baseline (all 49.4k training examples) |
| **E2** | Random 50% | Random | Uniform random subsampling |
| **E3** | Adaptive 50% | Random | Importance-scored selection (Diversity + Complexity + Length) |
| **E4** | Adaptive 50% | Curriculum | **Full Method** (Adaptive Selection + Easy→Hard Curriculum) |
| **E5** | Random 50% | Curriculum | Curriculum on random subset (Isolation test) |
| **Ablation A** | Adaptive 50% | Random | Diversity score only |
| **Ablation B** | Adaptive 50% | Random | Complexity score only |

**Hardware Target:** 1× NVIDIA GPU with $\ge$ 6–16 GB VRAM (e.g., T4, A100, RTX 3090/4090, L4).

In [2]:
!git clone https://github.com/Bezawit-cloud/efficient-llm-finetuning.git
%cd efficient-llm-finetuning

Cloning into 'efficient-llm-finetuning'...
remote: Enumerating objects: 64, done.
remote: Counting objects: 100% (64/64), done.
remote: Compressing objects: 100% (39/39), done.
remote: Total 64 (delta 24), reused 54 (delta 18), pack-reused 0 (from 0)
Receiving objects: 100% (64/64), 33.34 KiB | 16.67 MiB/s, done.
Resolving deltas: 100% (24/24), done.
/content/efficient-llm-finetuning/efficient-llm-finetuning


### Step 1: Environment & GPU Verification

In [3]:
# Verify GPU hardware
!nvidia-smi

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
    print(f"BF16 Support: {torch.cuda.is_bf16_supported()}")

Tue Aug 18 14:58:18 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

### Step 2: Install Dependencies

In [4]:
# Install repository requirements
%pip install -r requirements.txt -q

### Step 3: Run GPU Smoke Test
Verifies CUDA detection, Qwen2.5-0.5B loading, LoRA parameter attachment, 1 step execution, and peak memory logging.

In [5]:
!python src/gpu_smoke_test.py

Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
GPU MIGRATION SMOKE TEST — QWEN2.5-0.5B-INSTRUCT
PyTorch Version:     2.11.0+cu128
CUDA Available:      True
Active Device:       cuda (Tesla T4)
Total GPU VRAM:      14.56 GB
CUDA Version:        12.8
BF16 Supported:      True
-----------------------------------------------------------------
Loading Tokenizer:   Qwen/Qwen2.5-0.5B-Instruct ...
Map: 100% 16/16 [00:00<00:00, 2006.90 examples/s]
Loading Model:       Qwen/Qwen2.5-0.5B-Instruct on cuda ...
Loading weights: 100% 290/290 [00:00<00:00, 7428.44it/s]
Model loaded in:     1.12s
LoRA Attached:       4,399,104 trainable params (0

### Step 4: Generate Full 52k Importance Scoring Cache
Computes tri-component scores (Diversity + Complexity + Response Length) on GPU (~45–60s) and caches to `data/scored_alpaca.json`.

In [6]:
import json
from pathlib import Path
from src.utils import load_config, set_seed
from src.data_utils import load_alpaca_dataset
from src.scoring import score_dataset

config = load_config("configs/base_config.yaml")
set_seed(config["seed"])

cache_path = Path("data/scored_alpaca.json")
if not cache_path.exists():
    print("Loading Alpaca dataset for scoring...")
    train_ds, _ = load_alpaca_dataset(config)
    train_examples = [dict(ex) for ex in train_ds]
    print(f"Computing importance scores for {len(train_examples)} examples on GPU...")
    scored = score_dataset(train_examples, config)
    cache_path.parent.mkdir(parents=True, exist_ok=True)
    with open(cache_path, "w") as f:
        json.dump(scored, f)
    print(f"Scores successfully cached to {cache_path} ({len(scored)} items).")
else:
    print(f"Cached scores already exist at {cache_path}.")

Loading Alpaca dataset for scoring...


2026-08-18 14:59:01 | INFO     | data_utils | Loading dataset: tatsu-lab/alpaca
INFO:data_utils:Loading dataset: tatsu-lab/alpaca
2026-08-18 14:59:06 | INFO     | data_utils | Train: 49401, Eval: 2601
INFO:data_utils:Train: 49401, Eval: 2601
2026-08-18 14:59:13 | INFO     | scoring | Scoring 49401 examples (a=0.333, b=0.333, g=0.334)
INFO:scoring:Scoring 49401 examples (a=0.333, b=0.333, g=0.334)


Computing importance scores for 49401 examples on GPU...


2026-08-18 14:59:21 | INFO     | scoring | Computing embeddings with all-MiniLM-L6-v2 ...
INFO:scoring:Computing embeddings with all-MiniLM-L6-v2 ...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/193 [00:00<?, ?it/s]

2026-08-18 14:59:46 | INFO     | scoring | Importance score stats — mean: 0.6328, std: 0.0699, min: 0.3100, max: 0.9029
INFO:scoring:Importance score stats — mean: 0.6328, std: 0.0699, min: 0.3100, max: 0.9029


Scores successfully cached to data/scored_alpaca.json (49401 items).


### Step 5: Execute Primary Experiment Suite (E1 – E5)
Runs all experiments sequentially with fixed seed `42` and identical LoRA configuration.

In [7]:
# E1 — Full Baseline (100% data, random order, 3 epochs)
!python src/train_baseline.py --config configs/exp1_baseline.yaml

# E2 — Random 50% (random order, 3 epochs)
!python src/train_baseline.py --config configs/exp2_random50.yaml

# E3 — Adaptive 50% (random order, 3 epochs)
!python src/train_baseline.py --config configs/exp3_adaptive50_random.yaml

# E4 — Full Method: Adaptive 50% + Curriculum (easy->hard, 3 epochs)
!python src/train_baseline.py --config configs/exp4_adaptive50_curriculum.yaml

# E5 — Random 50% + Curriculum (isolation test, 3 epochs)
!python src/train_baseline.py --config configs/exp5_random50_curriculum.yaml

2026-08-18 15:00:44 | INFO     | train | === Experiment E1: full_baseline | seed=42 ===
2026-08-18 15:00:44 | INFO     | train | Config: configs/exp1_baseline.yaml
2026-08-18 15:00:45 | INFO     | data_utils | Loading dataset: tatsu-lab/alpaca
2026-08-18 15:00:47 | INFO     | data_utils | Train: 49401, Eval: 2601
2026-08-18 15:00:52 | INFO     | train | Loading cached scores from data/scored_alpaca.json
2026-08-18 15:00:52 | INFO     | select_and_order | Selection: method=full, fraction=1.0, n_total=49401
2026-08-18 15:00:52 | INFO     | select_and_order | Selected 49401 / 49401 examples (100.0%)
2026-08-18 15:00:52 | INFO     | train | Training on 49401 examples
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/lo

### Step 6: Execute Ablations (A & B)

In [8]:
# Ablation A — Diversity Only (50% subset, 3 epochs)
!python src/train_baseline.py --config configs/ablation_diversity_only.yaml

# Ablation B — Complexity Only (50% subset, 3 epochs)
!python src/train_baseline.py --config configs/ablation_complexity_only.yaml

2026-08-18 15:04:26 | INFO     | train | === Experiment A1: ablation_diversity_only | seed=42 ===
2026-08-18 15:04:26 | INFO     | train | Config: configs/ablation_diversity_only.yaml
2026-08-18 15:04:27 | INFO     | data_utils | Loading dataset: tatsu-lab/alpaca
2026-08-18 15:04:30 | INFO     | data_utils | Train: 49401, Eval: 2601
2026-08-18 15:04:34 | INFO     | train | Loading cached scores from data/scored_alpaca.json
2026-08-18 15:04:35 | INFO     | select_and_order | Selection: method=adaptive, fraction=0.5, n_total=49401
2026-08-18 15:04:35 | INFO     | select_and_order | Selected 24700 / 49401 examples (50.0%)
2026-08-18 15:04:35 | INFO     | train | Training on 24700 examples
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not loa

### Step 7: Results Compilation & Comparison Table

In [9]:
import glob
import pandas as pd

result_files = sorted(glob.glob("outputs/**/results.json", recursive=True))
records = []
for f in result_files:
    with open(f) as fp:
        records.append(json.load(fp))

if records:
    df = pd.DataFrame(records)
    cols = ["experiment_id", "experiment_name", "n_train_examples", "data_fraction", "ordering", "eval_loss", "train_loss", "wall_clock_minutes", "peak_gpu_memory_mb"]
    display_cols = [c for c in cols if c in df.columns]
    display(df[display_cols].sort_values("experiment_id"))
else:
    print("No results.json files found yet.")

No results.json files found yet.
